In [2]:
# ============================
# Carregando os arquivos CSV
# como Views temporárias
# ============================

con.execute("""
CREATE OR REPLACE VIEW src_usuarios AS
SELECT *
FROM read_csv_auto('sounddata_usuarios.csv');
""")

con.execute("""
CREATE OR REPLACE VIEW src_tracks AS
SELECT *
FROM read_csv_auto('sounddata_tracks.csv');
""")

con.execute("""
CREATE OR REPLACE VIEW src_artistas AS
SELECT *
FROM read_csv_auto('sounddata_artistas.csv');
""")

con.execute("""
CREATE OR REPLACE VIEW src_plays AS
SELECT *
FROM read_csv_auto('sounddata_plays.csv');
""")

print("✅ Views criadas com sucesso!")

✅ Views criadas com sucesso!


In [3]:
# Quantidade de registros em cada View

print("Usuários :", con.execute("SELECT COUNT(*) FROM src_usuarios").fetchone()[0])

print("Tracks :", con.execute("SELECT COUNT(*) FROM src_tracks").fetchone()[0])

print("Artistas :", con.execute("SELECT COUNT(*) FROM src_artistas").fetchone()[0])

print("Plays :", con.execute("SELECT COUNT(*) FROM src_plays").fetchone()[0])

Usuários : 30
Tracks : 31
Artistas : 21
Plays : 170


In [4]:
# Visualizando os primeiros registros das tabelas de origem

print("===== USUÁRIOS =====")
display(con.execute("SELECT * FROM src_usuarios LIMIT 5").df())

print("===== TRACKS =====")
display(con.execute("SELECT * FROM src_tracks LIMIT 5").df())

print("===== ARTISTAS =====")
display(con.execute("SELECT * FROM src_artistas LIMIT 5").df())

print("===== PLAYS =====")
display(con.execute("SELECT * FROM src_plays LIMIT 5").df())

===== USUÁRIOS =====


,usuario_id,nome,email,plano,pais,cidade,data_cadastro,faixa_etaria
0,USR0001,Aisha Ferreira,aisha.ferreira@email.com,premium_mensal,BR,Brasília,2021-02-02,55+
1,USR0002,Bruno Nakamura,bruno.nakamura@email.com,free,BR,São Paulo,2021-08-12,35-44
2,USR0003,Camila Souza,camila.souza@email.com,premium_mensal,BR,Curitiba,2022-09-09,25-34
3,USR0004,Diego Martins,diego.martins@email.com,premium_mensal,BR,São Paulo,2021-07-23,18-24
4,USR0005,Elena Costa,elena.costa@email.com,premium_anual,PT,Lisboa,2022-12-20,25-34


===== TRACKS =====


,track_id,titulo,artista_id,album,genero,duracao_seg,data_lancamento,explicit
0,TRK0001,Goela Abaixo,ART0001,Goela Abaixo,soul,213,2022-03-10,False
1,TRK0002,Ismael,ART0001,Nha Cretcheu,soul,187,2018-06-01,False
2,TRK0003,Ocas do Mundo,ART0002,Ocas do Mundo,pop,198,2023-01-20,False
3,TRK0004,Troca de Calçada,ART0002,Ocas do Mundo,pop,224,2023-01-20,False
4,TRK0005,Coentro,ART0003,Versions,pop,215,2023-08-11,False


===== ARTISTAS =====


,artista_id,nome,pais_origem,genero_prim,gravadora,seguidores
0,ART0001,Liniker,BR,soul,Warner,412000
1,ART0002,Jovem Dionisio,BR,pop,Universal,389000
2,ART0003,Gloria Groove,BR,pop,Sony,540000
3,ART0004,Djonga,BR,hip-hop,independente,290000
4,ART0005,Alceu Valenca,BR,mpb,Warner,178000


===== PLAYS =====


,play_id,usuario_id,track_id,artista_id,play_timestamp,duracao_seg,dispositivo,pais
0,PLY00001,USR0024,TRK0018,ART0013,2024-05-05 05:29:00,187,tablet,US
1,PLY00002,USR0022,TRK0008,ART0004,2024-06-15 01:14:00,166,smart_tv,AR
2,PLY00003,USR0009,TRK0013,ART0008,2024-02-03 06:58:00,272,tablet,AR
3,PLY00004,USR0021,TRK0007,ART0004,2024-09-12 12:56:00,230,desktop,BR
4,PLY00005,USR0005,TRK0009,ART0005,2024-05-06 23:35:00,211,tablet,PT


In [5]:
# ======================================
# Criando a dimensão de usuários
# ======================================

con.execute("""
CREATE OR REPLACE TABLE dim_usuario (

    sk_usuario INTEGER PRIMARY KEY,

    usuario_id VARCHAR NOT NULL,

    plano VARCHAR,

    faixa_etaria VARCHAR,

    pais VARCHAR,

    cidade VARCHAR,

    dt_inicio DATE,

    dt_fim DATE,

    is_current BOOLEAN DEFAULT TRUE

)
""")

print("✅ Tabela dim_usuario criada com sucesso!")

✅ Tabela dim_usuario criada com sucesso!


In [6]:
# ======================================
# DIM_TRACK
# ======================================

con.execute("""
CREATE OR REPLACE TABLE dim_track (

    sk_track INTEGER PRIMARY KEY,
    track_id VARCHAR NOT NULL,
    titulo VARCHAR,
    album VARCHAR,
    genero VARCHAR,
    duracao_total_seg INTEGER,
    explicit BOOLEAN,

    dt_inicio DATE,
    dt_fim DATE,
    is_current BOOLEAN DEFAULT TRUE

)
""")

# ======================================
# DIM_ARTISTA
# ======================================

con.execute("""
CREATE OR REPLACE TABLE dim_artista (

    sk_artista INTEGER PRIMARY KEY,
    artista_id VARCHAR NOT NULL,
    nome VARCHAR,
    pais_origem VARCHAR,
    genero_prim VARCHAR,
    gravadora VARCHAR,

    dt_inicio DATE,
    dt_fim DATE,
    is_current BOOLEAN DEFAULT TRUE

)
""")

# ======================================
# DIM_DATA
# ======================================

con.execute("""
CREATE OR REPLACE TABLE dim_data (

    sk_data INTEGER PRIMARY KEY,
    data_completa DATE,
    ano INTEGER,
    trimestre INTEGER,
    mes INTEGER,
    nome_mes VARCHAR,
    dia_semana INTEGER,
    nome_dia VARCHAR,
    hora INTEGER,
    fim_de_semana BOOLEAN

)
""")

# ======================================
# FATO
# ======================================

con.execute("""
CREATE OR REPLACE TABLE fato_plays (

    sk_play INTEGER PRIMARY KEY,

    sk_usuario INTEGER,

    sk_track INTEGER,

    sk_artista INTEGER,

    sk_data INTEGER,

    duracao_seg INTEGER,

    dispositivo VARCHAR,

    pais_play VARCHAR

)
""")

print("✅ Todas as tabelas foram criadas!")

✅ Todas as tabelas foram criadas!


In [7]:
con.execute("SHOW TABLES").df()

,name
0,dim_artista
1,dim_data
2,dim_track
3,dim_usuario
4,fato_plays
5,src_artistas
6,src_plays
7,src_tracks
8,src_usuarios


In [8]:
# ==========================================================
# Carregando DIM_USUARIO
# ==========================================================

con.execute("""
INSERT INTO dim_usuario
SELECT
    row_number() OVER() AS sk_usuario,
    usuario_id,
    plano,
    faixa_etaria,
    pais,
    cidade,
    CURRENT_DATE AS dt_inicio,
    DATE '9999-12-31' AS dt_fim,
    TRUE AS is_current
FROM src_usuarios;
""")

# ==========================================================
# Carregando DIM_TRACK
# ==========================================================

con.execute("""
INSERT INTO dim_track
SELECT
    row_number() OVER() AS sk_track,
    track_id,
    titulo,
    album,
    genero,
    duracao_seg,
    explicit,
    CURRENT_DATE,
    DATE '9999-12-31',
    TRUE
FROM src_tracks;
""")

# ==========================================================
# Carregando DIM_ARTISTA
# ==========================================================

con.execute("""
INSERT INTO dim_artista
SELECT
    row_number() OVER() AS sk_artista,
    artista_id,
    nome,
    pais_origem,
    genero_prim,
    gravadora,
    CURRENT_DATE,
    DATE '9999-12-31',
    TRUE
FROM src_artistas;
""")

print("✅ Dimensões carregadas com sucesso!")

✅ Dimensões carregadas com sucesso!


In [9]:
print("Usuários :", con.execute("SELECT COUNT(*) FROM dim_usuario").fetchone()[0])

print("Tracks :", con.execute("SELECT COUNT(*) FROM dim_track").fetchone()[0])

print("Artistas :", con.execute("SELECT COUNT(*) FROM dim_artista").fetchone()[0])

Usuários : 30
Tracks : 31
Artistas : 21


In [10]:
# ==========================================================
# Carregando DIM_DATA
# ==========================================================

INSERT_DIM_DATA = """
INSERT INTO dim_data
SELECT
    row_number() OVER() AS sk_data,

    CAST(play_timestamp AS DATE) AS data_completa,

    YEAR(play_timestamp) AS ano,

    QUARTER(play_timestamp) AS trimestre,

    MONTH(play_timestamp) AS mes,

    monthname(play_timestamp) AS nome_mes,

    isodow(play_timestamp) AS dia_semana,

    dayname(play_timestamp) AS nome_dia,

    HOUR(play_timestamp) AS hora,

    CASE
        WHEN isodow(play_timestamp) IN (6,7)
        THEN TRUE
        ELSE FALSE
    END AS fim_de_semana

FROM (

    SELECT DISTINCT play_timestamp

    FROM src_plays

);
"""

con.execute(INSERT_DIM_DATA)

print("✅ dim_data carregada!")

✅ dim_data carregada!


In [11]:
con.execute("""
SELECT *
FROM dim_data
LIMIT 5
""").df()

,sk_data,data_completa,ano,trimestre,mes,nome_mes,dia_semana,nome_dia,hora,fim_de_semana
0,1,2024-01-28,2024,1,1,January,7,Sunday,21,True
1,2,2024-10-05,2024,4,10,October,6,Saturday,22,True
2,3,2024-04-14,2024,2,4,April,7,Sunday,21,True
3,4,2024-10-10,2024,4,10,October,4,Thursday,4,False
4,5,2024-02-17,2024,1,2,February,6,Saturday,9,True


In [12]:
# ==========================================================
# Carregando a FATO
# ==========================================================

con.execute("""

INSERT INTO fato_plays

SELECT

    row_number() OVER() AS sk_play,

    u.sk_usuario,

    t.sk_track,

    a.sk_artista,

    d.sk_data,

    p.duracao_seg,

    p.dispositivo,

    p.pais

FROM src_plays p

JOIN dim_usuario u
ON u.usuario_id = p.usuario_id
AND u.is_current = TRUE

JOIN dim_track t
ON t.track_id = p.track_id
AND t.is_current = TRUE

JOIN dim_artista a
ON a.artista_id = p.artista_id
AND a.is_current = TRUE

JOIN dim_data d
ON d.data_completa = CAST(p.play_timestamp AS DATE)

""")

print("✅ fato_plays carregada!")

✅ fato_plays carregada!


In [13]:
print(
con.execute("""
SELECT COUNT(*)
FROM fato_plays
""").fetchone()
)

(278,)


In [14]:
con.execute("DELETE FROM dim_data")
con.execute("DELETE FROM fato_plays")

In [15]:
con.execute("""

INSERT INTO dim_data

SELECT

row_number() OVER() AS sk_data,

data_completa,

YEAR(data_completa),

QUARTER(data_completa),

MONTH(data_completa),

monthname(data_completa),

isodow(data_completa),

dayname(data_completa),

0,

CASE
WHEN isodow(data_completa) IN (6,7)
THEN TRUE
ELSE FALSE
END

FROM (

SELECT DISTINCT

CAST(play_timestamp AS DATE) AS data_completa

FROM src_plays

)

""")

In [16]:
con.execute("""
SELECT COUNT(*)
FROM fato_plays
""").fetchone()

(0,)

In [17]:
print("Dim Data:", con.execute("SELECT COUNT(*) FROM dim_data").fetchone()[0])

print("Fato:", con.execute("SELECT COUNT(*) FROM fato_plays").fetchone()[0])

Dim Data: 126
Fato: 0


In [18]:
print("Plays origem:",
      con.execute("SELECT COUNT(*) FROM src_plays").fetchone()[0])

print("Join usuário:",
      con.execute("""
SELECT COUNT(*)
FROM src_plays p
JOIN dim_usuario u
ON p.usuario_id = u.usuario_id
AND u.is_current = TRUE
""").fetchone()[0])

print("Join track:",
      con.execute("""
SELECT COUNT(*)
FROM src_plays p
JOIN dim_track t
ON p.track_id = t.track_id
AND t.is_current = TRUE
""").fetchone()[0])

print("Join artista:",
      con.execute("""
SELECT COUNT(*)
FROM src_plays p
JOIN dim_artista a
ON p.artista_id = a.artista_id
AND a.is_current = TRUE
""").fetchone()[0])

print("Join data:",
      con.execute("""
SELECT COUNT(*)
FROM src_plays p
JOIN dim_data d
ON CAST(p.play_timestamp AS DATE) = d.data_completa
""").fetchone()[0])

Plays origem: 170
Join usuário: 170
Join track: 170
Join artista: 170
Join data: 170


In [19]:

con.execute("DELETE FROM fato_plays")

In [20]:
con.execute("""
INSERT INTO fato_plays
SELECT
    row_number() OVER() AS sk_play,
    u.sk_usuario,
    t.sk_track,
    a.sk_artista,
    d.sk_data,
    p.duracao_seg,
    p.dispositivo,
    p.pais AS pais_play
FROM src_plays p
JOIN dim_usuario u
    ON u.usuario_id = p.usuario_id
   AND u.is_current = TRUE
JOIN dim_track t
    ON t.track_id = p.track_id
   AND t.is_current = TRUE
JOIN dim_artista a
    ON a.artista_id = p.artista_id
   AND a.is_current = TRUE
JOIN dim_data d
    ON d.data_completa = CAST(p.play_timestamp AS DATE);
""")

print("✅ fato_plays carregada!")

✅ fato_plays carregada!


In [21]:
con.execute("SELECT COUNT(*) FROM fato_plays").fetchone()

(170,)

In [22]:
result = con.execute("""
SELECT

    t.genero,

    COUNT(*) AS total_plays

FROM fato_plays f

JOIN dim_track t

ON t.sk_track = f.sk_track

GROUP BY t.genero

ORDER BY total_plays DESC
""")

result.df()

,genero,total_plays
0,hip-hop,50
1,pop,31
2,eletronica,21
3,soul,17
4,mpb,13
5,psychedelic,11
6,indie,9
7,rock,9
8,r&b,7
9,jazz,2


In [23]:
result = con.execute("""

SELECT

    u.plano,

    COUNT(*) AS total_plays,

    ROUND(AVG(f.duracao_seg),0) AS media_segundos,

    ROUND(AVG(f.duracao_seg)/60,1) AS media_minutos

FROM fato_plays f

JOIN dim_usuario u

ON u.sk_usuario = f.sk_usuario

GROUP BY u.plano

ORDER BY media_segundos DESC

""")

result.df()

,plano,total_plays,media_segundos,media_minutos
0,premium_anual,8,170.0,2.8
1,estudante,44,166.0,2.8
2,free,61,156.0,2.6
3,premium_mensal,57,152.0,2.5


In [1]:
import duckdb
import pandas as pd

# Cria ou abre o banco de dados
con = duckdb.connect("sounddata.duckdb")

print("✅ Banco conectado com sucesso!")

✅ Banco conectado com sucesso!


In [24]:
result = con.execute("""

SELECT

    a.nome AS artista,

    COUNT(DISTINCT f.sk_usuario) AS ouvintes_unicos,

    COUNT(*) AS total_plays

FROM fato_plays f

JOIN dim_artista a

ON a.sk_artista = f.sk_artista

GROUP BY a.nome

ORDER BY ouvintes_unicos DESC

LIMIT 5

""")

result.df()

,artista,ouvintes_unicos,total_plays
0,Liniker,15,17
1,Baco Exu do Blues,14,18
2,Jovem Dionisio,13,14
3,Matuê,10,12
4,Ana Frango Eletrico,9,12


In [25]:
result = con.execute("""

SELECT

    u.faixa_etaria,

    f.dispositivo,

    COUNT(*) AS total_plays

FROM fato_plays f

JOIN dim_usuario u

ON u.sk_usuario = f.sk_usuario

GROUP BY

u.faixa_etaria,

f.dispositivo

ORDER BY

u.faixa_etaria,

total_plays DESC

""")

result.df()

,faixa_etaria,dispositivo,total_plays
0,18-24,smartphone,21
1,18-24,desktop,18
2,18-24,tablet,10
3,18-24,smart_tv,7
4,25-34,desktop,19
5,25-34,smartphone,18
6,25-34,smart_tv,9
7,25-34,tablet,4
8,35-44,smartphone,11
9,35-44,desktop,11


In [26]:
# Fechar a versão atual
con.execute("""
UPDATE dim_usuario
SET
    dt_fim = DATE '2024-04-01',
    is_current = FALSE
WHERE usuario_id = 'USR0042'
AND is_current = TRUE;
""")

# Inserir a nova versão
con.execute("""
INSERT INTO dim_usuario
SELECT
    (SELECT MAX(sk_usuario)+1 FROM dim_usuario),
    usuario_id,
    'premium_mensal',
    faixa_etaria,
    pais,
    cidade,
    DATE '2024-04-01',
    DATE '9999-12-31',
    TRUE
FROM dim_usuario
WHERE usuario_id='USR0042'
AND is_current=FALSE
ORDER BY dt_fim DESC
LIMIT 1;
""")

In [27]:
con.execute("""
SELECT
    sk_usuario,
    usuario_id,
    plano,
    dt_inicio,
    dt_fim,
    is_current
FROM dim_usuario
WHERE usuario_id='USR0042'
ORDER BY dt_inicio
""").df()

,sk_usuario,usuario_id,plano,dt_inicio,dt_fim,is_current
0,31,USR0042,premium_mensal,2024-04-01,9999-12-31,True
1,28,USR0042,free,2026-07-15,2024-04-01,False


In [28]:
usuarios = [
    ("USR0117", "premium_anual", "2024-06-15"),
    ("USR0203", "free", "2024-09-01")
]

for usuario, novo_plano, data in usuarios:

    con.execute(f"""
    UPDATE dim_usuario
    SET
        dt_fim = DATE '{data}',
        is_current = FALSE
    WHERE usuario_id = '{usuario}'
      AND is_current = TRUE;
    """)

    con.execute(f"""
    INSERT INTO dim_usuario
    SELECT
        (SELECT MAX(sk_usuario)+1 FROM dim_usuario),
        usuario_id,
        '{novo_plano}',
        faixa_etaria,
        pais,
        cidade,
        DATE '{data}',
        DATE '9999-12-31',
        TRUE
    FROM dim_usuario
    WHERE usuario_id = '{usuario}'
      AND is_current = FALSE
    ORDER BY dt_fim DESC
    LIMIT 1;
    """)

print("✅ Usuários atualizados com SCD Tipo 2!")

✅ Usuários atualizados com SCD Tipo 2!


In [29]:
con.execute("""
SELECT
    usuario_id,
    plano,
    dt_inicio,
    dt_fim,
    is_current
FROM dim_usuario
WHERE usuario_id IN ('USR0042','USR0117','USR0203')
ORDER BY usuario_id, dt_inicio;
""").df()

,usuario_id,plano,dt_inicio,dt_fim,is_current
0,USR0042,premium_mensal,2024-04-01,9999-12-31,True
1,USR0042,free,2026-07-15,2024-04-01,False
2,USR0117,premium_anual,2024-06-15,9999-12-31,True
3,USR0117,premium_mensal,2026-07-15,2024-06-15,False
4,USR0203,free,2024-09-01,9999-12-31,True
5,USR0203,estudante,2026-07-15,2024-09-01,False


In [30]:
# Fechar a versão atual
con.execute("""
UPDATE dim_artista
SET
    dt_fim = DATE '2024-07-01',
    is_current = FALSE
WHERE artista_id = 'ART0019'
AND is_current = TRUE;
""")

# Criar nova versão
con.execute("""
INSERT INTO dim_artista
SELECT
    (SELECT MAX(sk_artista)+1 FROM dim_artista),
    artista_id,
    nome,
    pais_origem,
    genero_prim,
    'independente',
    DATE '2024-07-01',
    DATE '9999-12-31',
    TRUE
FROM dim_artista
WHERE artista_id='ART0019'
AND is_current=FALSE
ORDER BY dt_fim DESC
LIMIT 1;
""")

In [31]:
con.execute("""
UPDATE dim_artista
SET genero_prim = 'metal'
WHERE artista_id = 'ART0031';
""")

In [32]:
con.execute("""
SELECT
    artista_id,
    nome,
    gravadora,
    genero_prim,
    dt_inicio,
    dt_fim,
    is_current
FROM dim_artista
WHERE artista_id IN ('ART0019','ART0031')
ORDER BY artista_id, dt_inicio;
""").df()

,artista_id,nome,gravadora,genero_prim,dt_inicio,dt_fim,is_current
0,ART0019,Baco Exu do Blues,independente,hip-hop,2024-07-01,9999-12-31,True
1,ART0019,Baco Exu do Blues,Sony,hip-hop,2026-07-15,2024-07-01,False
2,ART0031,Torture Squad,independente,metal,2026-07-15,9999-12-31,True


In [33]:
result = con.execute("""
SELECT
    u.plano,
    COUNT(*) AS total_plays,
    ROUND(AVG(f.duracao_seg), 0) AS media_seg_por_play,
    ROUND(AVG(f.duracao_seg) / 60, 1) AS media_min_por_play
FROM fato_plays f
JOIN dim_usuario u
    ON u.sk_usuario = f.sk_usuario
GROUP BY u.plano
ORDER BY media_min_por_play DESC;
""")

result.df()

,plano,total_plays,media_seg_por_play,media_min_por_play
0,estudante,44,166.0,2.8
1,premium_anual,8,170.0,2.8
2,free,61,156.0,2.6
3,premium_mensal,57,152.0,2.5
